# Social Network Ads: Purchase Prediction
Predict whether a user will purchase a product from **Gender, Age and Estimated Salary** using a Random Forest classifier. The scaler and model are saved together as one Scikit-learn `Pipeline`, so the Flask app scales inputs automatically.

In [ ]:
import pandas as pd
import pickle
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## Load data

In [ ]:
# Change the path if needed (in Google Colab use '/content/Social_Network_Ads.csv')
df = pd.read_csv('Social_Network_Ads.csv')
df.head()

## EDA

In [ ]:
df.info()

In [ ]:
# Class balance of the target
df['Purchased'].value_counts()

In [ ]:
df.describe()

## Pre-processing

In [ ]:
# Check for null values
df.isna().sum()

In [ ]:
# Encode Gender (same encoding as the web form: Male = 1, Female = 0)
df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})
df.head()

## Select features and split
`User ID` is only an identifier with no predictive meaning, so it is **dropped**. The feature order here must match the order used in `app.py`.

In [ ]:
X = df[['Gender', 'Age', 'EstimatedSalary']]
y = df['Purchased']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)

## Build and train the pipeline

In [ ]:
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(n_estimators=100, criterion='entropy', random_state=0))
])

pipe.fit(X_train, y_train)

## Evaluation

In [ ]:
y_pred = pipe.predict(X_test)
print('Test accuracy:', accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred))
print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred))

In [ ]:
# 5-fold cross-validation gives a more reliable estimate than a single 80-row test set
cv_scores = cross_val_score(pipe, X, y, cv=5, scoring='accuracy')
print('CV accuracy per fold:', cv_scores.round(3))
print('Mean CV accuracy:', round(cv_scores.mean(), 3))

## Save the pipeline

In [ ]:
with open('model.pkl', 'wb') as f:
    pickle.dump(pipe, f)

## Test the saved model (same input format as the Flask app)

In [ ]:
with open('model.pkl', 'rb') as f:
    loaded = pickle.load(f)

sample = pd.DataFrame([[1, 35, 60000]], columns=['Gender', 'Age', 'EstimatedSalary'])
print('Will purchase' if loaded.predict(sample)[0] == 1 else 'Will not purchase')